In [10]:
from nimber_ops import *
from transfinite import w, Ordinal

In [25]:
def ord_decomp(ordinal : Ordinal) -> list:
    ''' returns [infinite, finite] where ordinal = infinite + finite'''
    high = Ordinal(ordinal.exponent, ordinal.coefficient)
    remainder = ordinal.addend
    while isinstance(remainder, Ordinal):
        high += Ordinal(remainder.exponent, remainder.coefficient, 0)
        remainder = remainder.addend
    assert(isinstance(remainder, int))
    return [high, remainder]

In [ ]:
class Nim:
    ''' nimbers '''
    def __init__(self, n : int | Ordinal) -> None:
        ''' ordinal considered an a field element in On_2 
        val = ordinal
        field = smallest x > n such that x is a field
        base = largest y < n such that y is a field (or base = 0 for n < 2)
        write n = high * base + low, where low,high < base
        '''
        def exp2(level : int) -> int:
            return 1 << level
        if isinstance(n, int):
            self.val = abs(n)
            self.isfinite = True            
            if self.val < 2:
                self.field = 2 # smallest 
                self.base = 0
                self.high = 0
                self.low = self.val
            else:
                level = 0
                while n >> (1 << level) > 0:
                    level += 1
                self.field = exp2(exp2(level))
                self.base = exp2(exp2(level - 1))
                self.high = self.val // self.base
                self.low = self.val - self.high * self.base
        else:
            assert isinstance(n, Ordinal), 'An infinite nimber must be an ordinal'
            self.val = n
            self.isfinite = False
            # find the smallest field containing n
            if (k:=n.exponent) < Ordinal(): # n = omega^k, k fintie => cubic extension
                power = 0
                while 3 ** power <= k: 
                    power += 1
                self.field = Ordinal(3**power)
            exp = 3**(power-1)
            self.base = Ordinal(exp)
            high = Ordinal(n.exponent - exp, n.coefficient, 0) if exp < n.exponent else n.coefficient
            remainder = n.addend
            while isinstance(remainder, Ordinal) and remainder.exponent > exp:
                high += Ordinal(remainder.exponent - exp, remainder.coefficient, 0)
                remainder = remainder.addend
                if remainder.exponent == exp: 
                    high += remainder.coefficient
                    remainder = remainder.addend
            self.high = high
            self.low = remainder
    
    def _repr_latex_(self):
        """
        Special method for Jupyter to render LaTeX.
        """
        if self.isfinite:
            # No special LaTeX for integers, just return the string
            return f"${self.val}$"
        else:
            # Delegate to the Ordinal's LaTeX representation
            return self.val._repr_latex_()
            
    def __repr__(self) -> str:
        if self.isfinite:       
            return str(self.val)
        else:
            return self.val.__repr__()
    
    def __add__(self, other):
        if self.isfinite & other.isfinite:
            return Nim(self.val ^ other.val)
        elif self.isfinite & (not other.isfinite):
            other_ord = other.val
            high = Ordinal(other_ord.exponent, other_ord.coefficient)
            remainder = other_ord.addend
            while isinstance(remainder, Ordinal):
                high += Ordinal(remainder.exponent, remainder.coefficient, 0)
                remainder = remainder.addend
            assert(isinstance(remainder, int))
            fin_part = self.val ^ remainder
            return Nim(high + fin_part)
        elif (not self.isfinite) & other.isfinite:
            return other + self
        else: # both infinite
            ord1, ord2 = self.val, other.val
            assert isinstance(ord1, Ordinal) and isinstance(ord2, Ordinal)
            
    
    def __mul__(self, other):
        x, y = self.val, other.val
        if self.isfinite and other.isfinite:
            def nim_product(a : int, b : int) -> int:
                # first handle trivial cases
                if a == 0 or b == 0:
                    return 0
                elif a == 1:
                    return b
                elif b == 1:
                    return a
                elif a == 2 and b == 2:
                    return 3
                else:
                    # do euclidean division by greatest possible fermat power 
                    # a = q_a * F_a + r_a and b = q_b * F_b + r_b
                    F_a, q_a, r_a = Nim(a).base, Nim(a).high, Nim(a).low
                    F_b, q_b, r_b = Nim(b).base, Nim(b).high, Nim(b).low
                    
                    # if one the Fermat powers is greater than the other, then
                    # nim multiplication by it is the same as ordinary multiplication
                    if F_a < F_b:
                        return nim_product(a,q_b)*F_b ^ nim_product(a,r_b)
                    elif F_a > F_b:
                        return nim_product(q_a,b)*F_a ^ nim_product(r_a,b)
                    else:
                        # otherwise we have to distribute and use F_n ** 2 = 3 * F_n / 2
                        p_1 = nim_product(q_a,q_b)
                        p_2 = nim_product(r_a,r_b)
                        p_3 = nim_product(q_a ^ r_a, q_b ^ r_b)
                        p_4 = nim_product(p_1, F_a >> 1)
                        p_5 = p_3 ^ p_2
                        return p_5 * F_a ^ p_2 ^ p_4
            return Nim(nim_product(x, y))  
        
    def sqrt(self):
        # if self.isfinite:
        if self.field == 2:
            return self
        term = self**2 + self
        return term.sqrt() + self
    
    def order(self):
        if self.isfinite:
            def fermat_divisors(n : int, include_one : bool = False ) -> list:
                '''
                Find the divisors of a Mersenne number 2 ** (2 ** n) - 1
                By default does not include 1
                '''
                # for now, this only works for n < 6
                # could potentiall go up to n = 11 using known factors on wikipedia 
                # no one knows the factors of 2 ** (2 ** 11) + 1
                if n >= 6:
                    raise ValueError('This function only works for n < 6')
                else:
                    divisors = []
                    for i in range(0 + int(not include_one),2 ** n):
                        product = 1
                        for j in range(n):
                            if i >> j & 1:
                                product *= 2 ** (2 ** j) + 1
                        divisors.append(product)
                    return divisors
                
            n = self.val
            if n == 0:
                return 0
            elif n == 1:
                return 1
            elif n in {2, 3}:
                return 3
            elif n == 3:
                return 3
            elif n < 1 << (1 << 5):
                # make more efficient by only checking possible orders
                # use Lagrange's theorem
                # find the smallest field containing n i.e. smallest F_k > n
                exp = (n.bit_length() - 1).bit_length()
                # find the order of n must divide F_k - 1 which factors by difference of squares
                divisors = fermat_divisors(exp)
                for factor in divisors[:-1]:
                    if (Nim(n)**factor).val == 1:
                        return factor
                else:
                    return divisors[-1] 
            else:
                for factor in fermat_divisors(5):
                    if (Nim(n)**factor).val == 1:
                        return factor
                    # brute force: will probably loop forever
                    i = 1 << (1 << 5) + 1
                    while True:
                        if (Nim(i)**factor).val == 1:
                            return i
                        i += 2
        else:
            ... # inifite case is hard..
    def inv(self):
        return self ** (-1)
        
    def __pow__(self, p):
        """
        Compute x**n using exponentiation by squaring.

        """
        if self.isfinite:
            if p >= 0: # binary exponentiation by squaring
                result = Nim(1) 
                while p > 0:
                    if p & 1:
                        result = result * self
                    self = self * self
                    p >>= 1
                return result
            elif p == -1:
                if self.field == 2: return self
                a, b, F, f = Nim(self.high), Nim(self.low), Nim(self.base), Nim(self.base >> 1)
                det = (a + b)*b + a*a*f
                return det**(-1) * (a*F + (a+b))
            else:
                inv = self ** (-1)
                return inv ** (-p)    

In [35]:
for n in range(5):
    F = 1<<(1<<n)
    x = Nim(F)
    ord_x = x.order()
    quot = (F**2 - 1)//ord_x
    print(f'order of {x} is {ord_x}, which is {quot}^th root of gen')
    n=0
    while ord_x < F**2-1:
        x = Nim(x.val + 1)
        ord_x = x.order()
        n+=1
    print(f'smallest generator of {F**2} is {x}, which is {n} more')

order of 2 is 3, which is 1^th root of gen
smallest generator of 4 is 2, which is 0 more
order of 4 is 15, which is 1^th root of gen
smallest generator of 16 is 4, which is 0 more
order of 16 is 85, which is 3^th root of gen
smallest generator of 256 is 18, which is 2 more
order of 256 is 21845, which is 3^th root of gen
smallest generator of 65536 is 258, which is 2 more
order of 65536 is 1431655765, which is 3^th root of gen
smallest generator of 4294967296 is 65540, which is 4 more


In [19]:
x = Nim(Ordinal(9)*2+Ordinal(2)+2)
y = Nim(7)
(y + x).__dict__

{'val': w**9*2 + w**2 + 5,
 'isfinite': False,
 'field': w**27,
 'base': w**9,
 'high': 2,
 'low': w**2 + 5}

In [7]:
# class Nim:
#     ''' finite nimbers '''
#     def __init__(self, n : int) -> None:
#         self.ord = abs(n)
#         # n = high * 2^(2 ^ (level-1)) + low;   high, low < 2^(2 ^ (level-1))
#         level = 0
#         if self.ord < 2:
#             self.level = level
#             self.fermat = None
#             self.high = 0
#             self.low = self.ord
#         else:
#             while n >> (1 << level) > 0:
#                 level += 1
#             self.level = level
#             self.fermat = (1 << (1 << (level - 1)))
#             self.high = self.ord // self.fermat
#             self.low = self.ord - self.high * self.fermat 
#     def __repr__(self) -> str:       
#         return str(self.ord)
    
#     def __add__(self, other):
#         return Nim(self.ord ^ other.ord)
    
#     def __mul__(self, other):
#         x = self.ord
#         y = other.ord
        
#         def nim_product(a : int, b : int) -> int:
#             # first handle trivial cases
#             if a == 0 or b == 0:
#                 return 0
#             elif a == 1:
#                 return b
#             elif b == 1:
#                 return a
#             elif a == 2 and b == 2:
#                 return 3
#             else:
#                 # do euclidean division by greatest possible fermat power 
#                 # a = q_a * F_a + r_a and b = q_b * F_b + r_b
#                 F_a, q_a, r_a = Nim(a).fermat, Nim(a).high, Nim(a).low
#                 F_b, q_b, r_b = Nim(b).fermat, Nim(b).high, Nim(b).low
                
#                 # if one the Fermat powers is greater than the other, then
#                 # nim multiplication by it is the same as ordinary multiplication
#                 if F_a < F_b:
#                     return nim_product(a,q_b)*F_b ^ nim_product(a,r_b)
#                 elif F_a > F_b:
#                     return nim_product(q_a,b)*F_a ^ nim_product(r_a,b)
#                 else:
#                     # otherwise we have to distribute and use F_n ** 2 = 3 * F_n / 2
#                     p_1 = nim_product(q_a,q_b)
#                     p_2 = nim_product(r_a,r_b)
#                     p_3 = nim_product(q_a ^ r_a, q_b ^ r_b)
#                     p_4 = nim_product(p_1, F_a >> 1)
#                     p_5 = p_3 ^ p_2
#                     return p_5 * F_a ^ p_2 ^ p_4
#         return Nim(nim_product(x, y))

In [8]:
def fermat_divisors(n : int, include_one : bool = False ) -> list:
                '''
                Find the divisors of a Mersenne number 2 ** (2 ** n) - 1
                By default does not include 1
                '''
                # for now, this only works for n < 6
                # could potentiall go up to n = 11 using known factors on wikipedia 
                # no one knows the factors of 2 ** (2 ** 11) + 1
                if n >= 6:
                    raise ValueError('This function only works for n < 6')
                else:
                    divisors = []
                    for i in range(0 + int(not include_one),2 ** n):
                        product = 1
                        for j in range(n):
                            if i >> j & 1:
                                product *= 2 ** (2 ** j) + 1
                        divisors.append(product)
                    return divisors